In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from src.data.market_loader import MarketLoader

from src.curves.curve_snapshot import CurveSnapshot
from src.curves.bootstrap.bootstrap_engine import BootstrapCurveEngine
from src.curves.projection_curve import ProjectionCurve
from src.curves.zero_curve import ZeroCurve
from src.curves.survival_curve import SurvivalCurve
from src.curves.simulator.hull_white_2factor import HullWhite2FactorSimulator
from src.curves.simulator.hull_white_2factor_pricer import HullWhite2FactorPricer

from src.instruments.instrument_builder import InstrumentBuilder

from src.trades.interest_rate_swap import InterestRateSwap

from src.pricing.hull_white_2factor_swap_pricer import HullWhite2FactorSwapPricer

from src.risk.exposure.stochastic_exposure_engine_2factor import MonteCarloExposureEngine2Factor

from src.xva.cva_engine import CVAEngine
from src.xva.dva_engine import DVAEngine
from src.xva.fva_engine import FVAEngine
from src.xva.colva_engine import ColVAEngine
from src.xva.mva_engine import MVAEngine
from src.xva.kva_engine import KVAEngine
from src.xva.xva_report import XVAReport

In [2]:
# downloading market curves
market_loader = MarketLoader()
market_curves = market_loader.market_loader_pipeline()

# downloading swap curves
swap_loader = MarketLoader()
swap_curves = swap_loader.swap_loader_pipeline()

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..
estr curve dataset already downloaded..
usd_ois curve dataset already downloaded..
eur_ois curve dataset already downloaded..


In [3]:
### create curve snapshots
# SOFR snapshot
sofr_df = market_curves['sofr']

latest_date = sofr_df.index[-1]
latest_sofr_curve = sofr_df.iloc[-1]

sofr_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'sofr',
    as_of_date = latest_date,
    curve_row = latest_sofr_curve
)

# Futures snapshot
future_df = market_curves['futures']
latest_futures_curve = future_df.iloc[-1]

futures_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'futures',
    as_of_date = latest_date,
    curve_row = latest_futures_curve
)

# OIS snapshot
ois_df = swap_curves['usd_ois']

latest_swap_date = ois_df.index[-1]
latest_swap_curve = ois_df.iloc[-1]

ois_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'usd_ois',
    as_of_date = latest_swap_date,
    curve_row = latest_swap_curve
)

In [4]:
### create instruments from curve snapshot
# deposits
deposit_instruments = InstrumentBuilder.build_deposit_instruments(snapshot = sofr_snapshot)

# futures
future_instruments = InstrumentBuilder.build_future_instruments(snapshot = futures_snapshot)

# ois
ois_instruments = InstrumentBuilder.build_ois_instruments(snapshot = ois_snapshot)

### discount, projection and zero curve builder
# bootstrapping engine for generating the discount curve
all_instruments = deposit_instruments + future_instruments + ois_instruments

engine = BootstrapCurveEngine()

discount_curve = engine.bootstrap(
    snapshot = sofr_snapshot,
    instruments = all_instruments
)

# projection curve
projection_curve = ProjectionCurve(discount_curve = discount_curve)

# zero curve
zero_curve = ZeroCurve(discount_curve = discount_curve)

In [5]:
# sample IRS trade objects
IR_swap = InterestRateSwap(
    notional = 1_000_000,
    maturity = 3.0,
    fixed_rate = 3.40,
    pay_fixed = True
)

In [6]:
### HW2F exposure risk analytics
# short rate
r0 = zero_curve.get_zero_rate(maturity = 0.25)

# hw simulator
hw_simulator = HullWhite2FactorSimulator(
    r0 = r0,
    random_seed = 2
)

# hw pricer
hw_pricer = HullWhite2FactorPricer(zero_curve = zero_curve)

# hw 2-factor swap pricer
hw_swap_pricer = HullWhite2FactorSwapPricer(hw_pricer = hw_pricer)

# hw 2-factor exposure engine
mc_hw2f_engine = MonteCarloExposureEngine2Factor(
    pricer = hw_swap_pricer,
    simulator = hw_simulator
)

In [7]:
# survival curve
counterparty_survival_curve = SurvivalCurve(hazard_rate = 0.005)
own_survival_curve = SurvivalCurve(hazard_rate = 0.01)

In [8]:
# CVA framework - HW2F
cva_engine = CVAEngine(
    exposure_engine = mc_hw2f_engine,
    discount_curve = discount_curve,
    survival_curve = counterparty_survival_curve,
    recovery_rate = 0.4
)

cva_profile, cva = cva_engine.cva_profile(
    swap = IR_swap,
    n_paths = 250,
    steps_per_year = 4
)

print(f'Total CVA: {cva:.1f}')

cva_profile

Total CVA: 57.7


,Times,EE,DF,PD,CVA_Contribution
0,0.00,7858.425573,0.999999,0.000000,0.000000
1,0.25,9973.485781,0.991079,0.001249,7.408752
2,0.50,10470.307863,0.982141,0.001248,7.698045
3,0.75,10156.479085,0.972673,0.001246,7.386086
4,1.00,9207.764278,0.963206,0.001245,6.622690
5,1.25,9130.352023,0.954487,0.001243,6.499442
6,1.50,8731.886072,0.945769,0.001241,6.151326
7,1.75,7378.961374,0.937051,0.001240,5.143883
8,2.00,6126.808717,0.928333,0.001238,4.225983
9,2.25,4597.670638,0.920364,0.001237,3.140108


In [9]:
# DVA framework - HW2F
dva_engine = DVAEngine(
    exposure_engine = mc_hw2f_engine,
    discount_curve = discount_curve,
    survival_curve = own_survival_curve,
    recovery_rate = 0.4
)

dva_profile, dva = dva_engine.dva_profile(
    swap = IR_swap,
    n_paths = 250,
    steps_per_year = 4    
)

print(f'Total DVA: {dva:.1f}')

dva_profile

Total DVA: 77.5


,Times,ENE,DF,PD,DVA_Contribution
0,0.00,-0.000000,0.999999,0.000000,-0.000000
1,0.25,3002.923668,0.991079,0.002497,4.458626
2,0.50,5208.445948,0.982141,0.002491,7.644427
3,0.75,6281.588243,0.972673,0.002484,9.107802
4,1.00,6906.926345,0.963206,0.002478,9.892250
5,1.25,7019.235878,0.954487,0.002472,9.937235
6,1.50,6574.241148,0.945769,0.002466,9.199211
7,1.75,6146.751806,0.937051,0.002460,8.500470
8,2.00,5474.848593,0.928333,0.002454,7.482111
9,2.25,4121.985238,0.920364,0.002447,5.570944


In [10]:
# FVA framework - HW2F
fva_engine = FVAEngine(
    exposure_engine = mc_hw2f_engine,
    discount_curve = discount_curve,
    funding_spread = 0.005
)

fva_profile, fva = fva_engine.fva_profile(
    swap = IR_swap,
    n_paths = 250,
    steps_per_year = 4    
)

print(f'Total FVA: {fva:.1f}')

fva_profile

Total FVA: 112.0


,Times,EE,DF,dt,FVA_Contribution
0,0.00,7858.425573,0.999999,0.00,0.000000
1,0.25,10467.714033,0.991079,0.25,12.967913
2,0.50,12250.932744,0.982141,0.25,15.040185
3,0.75,11993.239884,0.972673,0.25,14.581883
4,1.00,11388.784378,0.963206,0.25,13.712175
5,1.25,10641.137095,0.954487,0.25,12.696038
6,1.50,9714.988233,0.945769,0.25,11.485170
7,1.75,8626.380977,0.937051,0.25,10.104198
8,2.00,6870.680329,0.928333,0.25,7.972847
9,2.25,5556.074386,0.920364,0.25,6.392016


In [11]:
# ColVA framework - HW2F
funding_rate = 0.050
collateral_rate = 0.045

colva_engine = ColVAEngine(
    exposure_engine = mc_hw2f_engine,
    discount_curve = discount_curve,
    funding_rate = funding_rate,
    collateral_rate = collateral_rate
)

colva_profile, colva = colva_engine.colva_profile(
    swap = IR_swap,
    n_paths = 250,
    steps_per_year = 4
)

print(f'Total ColVA: {colva:.1f}')

colva_profile

Total ColVA: 91.7


,Times,EE,DF,dt,ColVA_Contribution
0,0.00,7858.425573,0.999999,0.00,0.000000
1,0.25,9162.655261,0.991079,0.25,11.351142
2,0.50,10225.651360,0.982141,0.25,12.553794
3,0.75,9167.667402,0.972673,0.25,11.146433
4,1.00,8704.103337,0.963206,0.25,10.479801
5,1.25,8760.147873,0.954487,0.25,10.451813
6,1.50,7643.983357,0.945769,0.25,9.036804
7,1.75,7282.954788,0.937051,0.25,8.530624
8,2.00,5786.598591,0.928333,0.25,6.714861
9,2.25,4857.941421,0.920364,0.25,5.588845


In [12]:
# MVA framework - HW2F
funding_spread = funding_rate - collateral_rate
initial_margin_multiplier = 1.1
percentile = 95.0

mva_engine = MVAEngine(
    exposure_engine = mc_hw2f_engine,
    discount_curve = discount_curve,
    funding_spread = funding_spread,
    im_multiplier = initial_margin_multiplier,
    percentile = percentile
)

mva_profile, mva = mva_engine.mva_profile(
    swap = IR_swap,
    n_paths = 250,
    steps_per_year = 4
)

print(f'Total MVA: {mva:.1f}')

mva_profile

Total MVA: 365.9


,Times,InitialMargin,DF,dt,MVA_Contribution
0,0.00,8644.268130,0.999999,0.00,0.000000
1,0.25,31165.518808,0.991079,0.25,38.609358
2,0.50,35060.906199,0.982141,0.25,43.043458
3,0.75,35058.031183,0.972673,0.25,42.625020
4,1.00,34581.136675,0.963206,0.25,41.635928
5,1.25,41675.685211,0.954487,0.25,49.723642
6,1.50,35009.964737,0.945769,0.25,41.389180
7,1.75,27836.346054,0.937051,0.25,32.605092
8,2.00,23387.939668,0.928333,0.25,27.139737
9,2.25,19997.434912,0.920364,0.25,23.006156


In [13]:
# KVA framework - HW2F
required_capital = 0.08
cost_of_capital = 0.10

kva_engine = KVAEngine(
    exposure_engine = mc_hw2f_engine,
    discount_curve = discount_curve,
    capital_ratio = required_capital,
    cost_of_capital = cost_of_capital
)

kva_profile, kva = kva_engine.kva_profile(
    swap = IR_swap,
    n_paths = 250,
    steps_per_year = 4  
)

print(f'Total KVA: {kva:.1f}')

kva_profile

Total KVA: 162.5


,Times,EE,Capital,DF,dt,KVA_Contribution
0,0.00,7858.425573,628.674046,0.999999,0.00,0.000000
1,0.25,10244.443864,819.555509,0.991079,0.25,20.306103
2,0.50,10469.934897,837.594792,0.982141,0.25,20.565912
3,0.75,10048.808139,803.904651,0.972673,0.25,19.548418
4,1.00,10131.512704,810.521016,0.963206,0.25,19.517458
5,1.25,10237.453415,818.996273,0.954487,0.25,19.543039
6,1.50,9093.320242,727.465619,0.945769,0.25,17.200363
7,1.75,7818.892626,625.511410,0.937051,0.25,14.653401
8,2.00,6371.454556,509.716364,0.928333,0.25,11.829659
9,2.25,5217.275115,417.382009,0.920364,0.25,9.603587


In [14]:
# XVA reports
xva_engine = XVAReport(
    cva_engine = cva_engine,
    dva_engine = dva_engine,
    fva_engine = fva_engine,
    colva_engine = colva_engine,
    mva_engine = mva_engine,
    kva_engine = kva_engine
)

summary_report_xva = xva_engine.summary_report(
    swap = IR_swap,
    n_paths = 250,
    steps_per_year = 4
)

full_report_xva = xva_engine.full_report(
    swap = IR_swap,
    n_paths = 250,
    steps_per_year = 4
)

display(summary_report_xva)
display(full_report_xva)

,Metrics,Value
0,CVA,60.377803
1,DVA,66.556256
2,BVA,6.178453
3,FVA,80.978572
4,ColVA,95.641392
5,MVA,360.854625
6,KVA,148.364453
7,XVA,-679.660589


,Times,CVA_Contribution,DVA_Contribution,FVA_Contribution,ColVA_Contribution,MVA_Contribution,KVA_Contribution,BVA_Contribution,XVA_Contribution
0,0.00,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,-0.000000
1,0.25,7.429919,3.928448,10.587910,11.798492,40.833536,17.481216,-3.501471,-84.202625
2,0.50,7.191940,7.305904,12.178370,11.533230,51.720162,18.578616,0.113964,-93.896414
3,0.75,7.762180,9.011029,12.395025,11.846074,50.659687,17.371778,1.248849,-91.023715
4,1.00,6.791616,10.177888,10.456675,11.560654,50.006148,16.455168,3.386272,-85.092373
5,1.25,6.334081,10.528429,10.367984,10.107964,46.040836,15.485086,4.194349,-77.807521
6,1.50,5.658104,10.078567,9.346100,9.127350,42.666932,13.388219,4.420464,-70.108137
7,1.75,5.017537,9.640085,8.271319,7.962182,36.329142,12.554030,4.622548,-60.494125
8,2.00,4.093625,8.572018,6.442509,6.386178,29.616382,10.407020,4.478392,-48.373697
9,2.25,3.564256,6.394657,5.018206,5.121668,23.217542,8.523579,2.830401,-39.050594
